# Topic 4: AWS Glue Deep Dive

> **Phase 7 → Week 13 → Topic 4**

---

## What This Covers

Week 12 covered Glue at a high level. This notebook goes deep: Glue Data Catalog, Crawlers, Bookmarks, DynamicFrame API, Glue Studio, and the Glue vs EMR decision matrix.

---

## Interview Cheat Sheet

**Q: What is the Glue Data Catalog?**
> The Glue Data Catalog is AWS's central metadata repository — a managed Hive-compatible metastore. It stores database/table definitions, schemas, and partition metadata. Any service can query against it: EMR (as Hive metastore), Athena, Redshift Spectrum, Glue jobs. It replaces the need to run your own Hive metastore. Each AWS account gets one Glue Catalog per region.

**Q: What is a Glue Crawler?**
> A Glue Crawler scans data stores (S3, JDBC databases, DynamoDB) and automatically creates or updates table definitions in the Glue Catalog. It infers schema from file contents (Parquet, JSON, CSV), detects partition structure from S3 folder paths, and can be scheduled to run periodically. After a crawler runs, the table is immediately queryable by Athena without any additional setup.

**Q: What is a Glue Bookmark?**
> Glue Job Bookmarks track which S3 objects a Glue job has already processed. When a job with bookmarks enabled runs again, it only reads new/unprocessed files — like a built-in checkpoint for batch ETL. Bookmarks are stored by Glue and tied to the job name. Reset a bookmark to reprocess all data from scratch.

In [ ]:
print("AWS Glue Deep Dive — reference patterns")
print("Requires AWS Glue environment to execute.")

## Part 1: Glue Data Catalog

In [ ]:
print("""
Glue Data Catalog — Structure & Usage:
══════════════════════════════════════════════════════════════

Hierarchy:   Catalog (per-account-per-region)
               └── Database
                     └── Table
                           └── Partitions

Create a table manually (boto3):
  import boto3
  glue = boto3.client('glue', region_name='us-east-1')

  glue.create_table(
    DatabaseName='analytics',
    TableInput={
      'Name': 'orders',
      'StorageDescriptor': {
        'Location': 's3://my-bucket/data/orders/',
        'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
        'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
        'SerdeInfo': {'SerializationLibrary': 'org.apache.hive.hcatalog.data.JsonSerDe'},
        'Columns': [
          {'Name': 'order_id',   'Type': 'bigint'},
          {'Name': 'customer_id','Type': 'bigint'},
          {'Name': 'amount',     'Type': 'double'},
          {'Name': 'status',     'Type': 'string'},
        ]
      },
      'PartitionKeys': [{'Name': 'date', 'Type': 'string'}],  # s3://…/date=2024-01-15/
    }
  )

Query the catalog from Athena (no data movement):
  SELECT order_id, amount, status
  FROM analytics.orders
  WHERE date = '2024-01-15'
    AND status = 'shipped';

Query the catalog from EMR/Spark:
  spark.sql("SELECT * FROM analytics.orders WHERE date = '2024-01-15'")
  # Requires: --conf hive.metastore.client.factory.class=..AWSGlueDataCatalogHiveClientFactory

Why use the Glue Catalog?
  ✓ Single schema registry for ALL services (Athena, EMR, Glue, Redshift Spectrum)
  ✓ Partition pruning across services
  ✓ Schema version history
  ✓ Zero-cost for catalog queries (pay for S3 reads)
══════════════════════════════════════════════════════════════
""")

## Part 2: Glue Crawlers

In [ ]:
print("""
Glue Crawlers — Automated Schema Discovery:
══════════════════════════════════════════════════════════════

Create a crawler (boto3):
  glue.create_crawler(
    Name='orders-crawler',
    Role='arn:aws:iam::123:role/GlueRole',
    DatabaseName='analytics',
    Targets={
      'S3Targets': [{'Path': 's3://my-bucket/data/orders/'}]
    },
    Schedule='cron(0 6 * * ? *)',   # daily at 6am UTC
    SchemaChangePolicy={
      'UpdateBehavior': 'UPDATE_IN_DATABASE',   # update schema on change
      'DeleteBehavior': 'DEPRECATE_IN_DATABASE' # mark deleted tables deprecated
    },
    RecrawlPolicy={'RecrawlBehavior': 'CRAWL_NEW_FOLDERS_ONLY'}  # incremental
  )

Crawler behaviors:
  CRAWL_EVERYTHING:        re-scan all folders every run (slow, expensive)
  CRAWL_NEW_FOLDERS_ONLY:  only scan folders added since last run (fast)
  CRAWL_EVENT_MODE:        trigger on S3 event notifications (near real-time)

Schema change policies:
  UpdateBehavior:
    UPDATE_IN_DATABASE:    auto-update schema when new columns detected
    LOG:                   log changes but don't update
  DeleteBehavior:
    DELETE_FROM_DATABASE:  remove table if S3 path deleted
    DEPRECATE_IN_DATABASE: mark deprecated (safer)
    LOG:                   log only

Crawler partition detection:
  S3 path: s3://bucket/orders/year=2024/month=01/day=15/file.parquet
  Crawler detects: partition keys = year, month, day
  Creates Hive-style partitions in the Glue Catalog automatically

Run crawler manually:
  glue.start_crawler(Name='orders-crawler')
  # Wait until state = READY (not RUNNING)
  while glue.get_crawler(Name='orders-crawler')['Crawler']['State'] == 'RUNNING':
      time.sleep(10)
══════════════════════════════════════════════════════════════
""")

## Part 3: DynamicFrame API

In [ ]:
print("""
Glue DynamicFrame vs Spark DataFrame:
══════════════════════════════════════════════════════════════

DynamicFrame is Glue's schema-flexible data abstraction.
Each row can have different columns and types — useful for messy JSON/XML.

Key DynamicFrame operations:

# 1. Read from Glue Catalog
dyf = glueContext.create_dynamic_frame.from_catalog(
    database="analytics",
    table_name="orders",
    push_down_predicate="date='2024-01-15'"  # partition pruning
)

# 2. Resolve ambiguous columns (DynamicFrame handles mixed types)
dyf_resolved = dyf.resolveChoice(
    specs=[("price", "cast:double")]  # force 'price' to double
)
# OR: let Glue choose automatically
dyf_resolved = dyf.resolveChoice(choice="make_struct")  # mixed → struct
dyf_resolved = dyf.resolveChoice(choice="project:double")  # cast all to double

# 3. Apply a filter
from awsglue.transforms import Filter
valid = Filter.apply(frame=dyf, f=lambda row: row['amount'] > 0)

# 4. Map (transform each row)
from awsglue.transforms import Map
def add_region(record):
    record['region'] = 'US' if record.get('country') == 'USA' else 'INTL'
    return record
mapped = Map.apply(frame=dyf, f=add_region)

# 5. Convert to Spark DataFrame for complex operations
df = dyf.toDF()
df = df.groupBy("customer_id").agg(F.sum("amount").alias("total"))
dyf_out = DynamicFrame.fromDF(df, glueContext, "result")

# 6. Write to S3 via Glue
glueContext.write_dynamic_frame.from_options(
    frame=dyf_out,
    connection_type="s3",
    connection_options={"path": "s3://my-bucket/output/"},
    format="parquet",
    format_options={"compression": "snappy"}
)

DynamicFrame vs DataFrame:
  DynamicFrame: handles schema inconsistencies, has resolveChoice(),
                works with Glue Catalog natively, writes to S3 atomically
  DataFrame:    full Spark API, better for complex transformations, SQL,
                window functions, joins — convert with .toDF()

Best practice: read as DynamicFrame, resolve/clean, convert to DataFrame,
               do transformations, convert back, write via Glue.
══════════════════════════════════════════════════════════════
""")

## Part 4: Glue Job Bookmarks

In [ ]:
print("""
Glue Job Bookmarks — Incremental Processing:
══════════════════════════════════════════════════════════════

Bookmarks track which S3 files a Glue job has already processed.
On each run, the job only reads files added SINCE the last successful run.

Enable in job code:
  job.init(args['JOB_NAME'], args)  # args must include --job-bookmark-option
  # Glue tracks: which S3 ETag/timestamp was last processed

Enable via AWS Console / CLI when creating the job:
  aws glue create-job \\
    --name my-etl-job \\
    --role GlueRole \\
    --command '{"Name": "glueetl", "ScriptLocation": "s3://bucket/script.py"}' \\
    --default-arguments '{"--job-bookmark-option": "job-bookmark-enable"}'

Options:
  job-bookmark-enable:   track and skip already-processed files
  job-bookmark-disable:  process all files every run (full reload)
  job-bookmark-pause:    don't advance the bookmark (re-read same batch)

Reset a bookmark (to reprocess all data):
  aws glue reset-job-bookmark --job-name my-etl-job

Bookmark vs custom watermark:
  Bookmark:         Glue-managed, S3 file-level, opaque
  Custom watermark: store last processed timestamp in S3/DynamoDB,
                    filter with push_down_predicate on partition key
                    → more transparent, works across any system

Bookmark limitations:
  Only works with S3 sources (not JDBC)
  Only tracks file additions (not modifications)
  Tied to job name — renaming the job resets the bookmark
  Doesn't work with DynamicFrame.fromDF() → use only from_catalog/from_options
══════════════════════════════════════════════════════════════
""")

## Part 5: Glue vs EMR Decision Matrix

In [ ]:
print("""
Glue vs EMR — When to Choose Each:
══════════════════════════════════════════════════════════════

Criterion               Glue                        EMR
───────────────────── ──────────────────────────── ────────────────────────────
Team expertise        Low Spark expertise OK        Spark/Hadoop expertise needed
Cluster management    None (serverless)             Full (create/terminate/tune)
Spark version         Fixed (per release label)     Choose any version
Custom Spark configs  Limited (can't set all)       Full control
Streaming             Not supported                 Structured Streaming ✓
DPU pricing           $0.44/DPU-hr (expensive)     EC2 spot pricing (cheaper)
Job frequency         Sporadic (≤ 5/day)            High frequency (100+/day)
Data volume           < 1 TB/job                   Any size
Custom packages       S3 .whl upload                Bootstrap action (any)
Glue Catalog          Native integration            Needs config to use
Athena integration    Automatic                     Via Glue Catalog
Debugging             Limited logs                  Full Spark UI + logs

Choose Glue when:
  ✓ Small team, no Spark expertise
  ✓ Simple batch ETL: S3 → transform → S3/Redshift
  ✓ Don't want to manage cluster lifecycle
  ✓ Sporadic jobs (< 5 runs/day) where idle cluster cost > Glue DPU cost
  ✓ Need Glue Catalog integration with Athena as primary query engine

Choose EMR when:
  ✓ Need Structured Streaming
  ✓ Need precise Spark tuning (custom JVM args, Spark version)
  ✓ High-frequency jobs (cluster amortizes startup cost)
  ✓ Very large data (> 1 TB/run, cost-sensitive → Spot instances)
  ✓ Need custom Python packages not installable via Glue's whl mechanism
  ✓ Require Spark UI for debugging

Cost comparison (example: 4-hour job, 10 m5.2xlarge equivalent):
  Glue (10 DPUs):       10 × $0.44 × 4 = $17.60
  EMR On-Demand:        10 × $0.384 × 4 = $15.36 (+ EMR fee ~10%)
  EMR Spot (80% disc):  10 × $0.077 × 4 = $3.08
  → For large/frequent jobs: EMR Spot wins dramatically
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Design a data catalog strategy for a company with 50 S3 buckets across dev/staging/prod. How would you structure Glue databases, set up crawlers, and manage access across environments?
2. Write a complete Glue ETL script that: reads from the Glue Catalog with partition pushdown, resolves ambiguous column types with DynamicFrame, transforms to DataFrame for aggregation, and writes partitioned Parquet with job bookmarks.
3. A Glue Crawler runs on an S3 bucket with `year=2024/month=01/day=15/` folder structure. After adding new data for `day=16`, the crawler runs again. What happens to the Glue Catalog? Does Athena immediately see the new partition?
4. You have a Glue job that processes 500 GB of JSON data daily with inconsistent schemas (some records have extra fields). DynamicFrame's `resolveChoice` shows 3 conflicting types for the `price` column. What strategy would you use and why?
5. Compare the cost of running a daily 2-hour ETL job 365 days/year using: (a) Glue with 5 DPUs, (b) EMR with 5 On-Demand m5.xlarge, (c) EMR Serverless. Which wins and why?